# Stage C — tokenizer and CPU basal gate
Runs the five-tokenizer intrinsic study and tiny paper-MAC CPU comparison. Edit only the configuration cell.

In [ ]:
# USER CONFIGURATION
REPO_URL = 'https://github.com/SynBioDex/SeqTrainer.git'
GIT_REF = 'REPLACE_WITH_REVIEWED_COMMIT'
DRIVE_ROOT = '/content/drive/MyDrive/SeqTrainerStageC'
TRAIN_FASTA = f'{DRIVE_ROOT}/inputs/train.fa.gz'
VALIDATION_FASTA = f'{DRIVE_ROOT}/inputs/validation.fa.gz'
RUN_NAME = 'c1_tokenizers_cpu'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys
repo = Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)], check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'], check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan,evo2-tokenizer]'], check=True)
print(subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'], text=True).strip())

In [ ]:
from huggingface_hub import snapshot_download
dnabert = f'{DRIVE_ROOT}/tokenizers/dnabert2'
snapshot_download('zhihan1996/DNABERT-2-117M', local_dir=dnabert, allow_patterns=['tokenizer*','vocab*','special_tokens_map.json','tokenizer_config.json','config.json'])
output = f'{DRIVE_ROOT}/runs/{RUN_NAME}'
def logged(label, command):
    subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',output,'--label',label,'--repo',str(repo),'--',*command], check=True)
logged('tokenizer_study',['seqtrainer-titans-stage-c-tokenizers','--train-fasta',TRAIN_FASTA,'--validation-fasta',VALIDATION_FASTA,'--dnabert2-path',dnabert,'--output-dir',output])
logged('cpu_pilot',['seqtrainer-titans-stage-c-cpu-pilot','--train-fasta',TRAIN_FASTA,'--validation-fasta',VALIDATION_FASTA,'--dnabert2-path',dnabert,'--output-dir',output,'--steps','3'])

In [ ]:
import os
print('SHARE THIS DIRECTORY:', output)
print('\n'.join(sorted(os.listdir(output))))